# Week 10 · Session 2 · Part B — Is something changing?
## From anomalies to trends: Mann–Kendall, Sen's slope and maps of trends

**Geoprocesamiento · Maestría en Geomática · Universidad Nacional de Colombia**

Part A did **step 1**: it removed the seasonal cycle $S$ and looked at the anomaly. Part B does the other two steps:

| Step | Remove | What is left | Question |
|---|---|---|---|
| 2 | the noise $\varepsilon$ (average each year) | $I + T$ | **Is it changing?** → the slope of a line ($T$) |
| 3 | the trend $T$ (subtract the line) | $I$ | **Which years were different?** |

A trend has two questions, and each has its tool:

| Question | Tool | Answer |
|---|---|---|
| Is there a monotonic change, or could it be chance? | **Mann–Kendall test** | S, Kendall's τ and a p-value |
| How much does it change per year? | **Theil–Sen slope** | NDVI per year |

**Significant does not mean large, and large does not mean significant.** Always report both.

## 0 · Setup
**0.1 · The working folder** (the same check as Part A: Jupyter Lab must be opened from the course folder).

In [ ]:
# ---- Check the working folder (Jupyter Lab, Anaconda) ----
# Open Jupyter Lab FROM the course folder, the one that contains "src/ts_tools.py" and the .h5 files.
import os
print("Working folder:", os.getcwd())
print("Is the toolbox here?", os.path.exists("src/ts_tools.py"))

# If a library is missing, remove the "#" of ONE of these lines, run it once, then restart the kernel:
# %conda install -y -c conda-forge h5py scipy statsmodels folium pystac-client planetary-computer odc-stac
# %pip install h5py scipy statsmodels folium pystac-client planetary-computer odc-stac pymannkendall

# Autoreload: if you edit src/ts_tools.py, the notebook uses the new version without restarting the kernel.
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np                  # arrays and maths
import pandas as pd                 # tables and dates
import matplotlib.pyplot as plt     # figures
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)      # "mean of empty slice" with missing data: expected
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
from src import ts_tools as tt      # our toolbox (src/ts_tools.py)

MODIS_FILE = "saldana_modis_ndvi_2001_2015_terra_aqua.h5"     # the same file as Part A (use the same name you used there)
TREND_FILE = "saldana_modis_trends_2001_2015.h5"              # the result we save at the end
P = 46                                                        # composites per year
REF_YEARS = (2001, 2015)

cube = tt.load_cube_hdf5(MODIS_FILE)     # read the cube saved in Part A
ndvi = cube["ndvi"]                      # NDVI values: (time, rows, columns); NaN = no reliable value
dates = pd.DatetimeIndex(cube["time"])   # the date of each layer
print("Shape (time, rows, columns):", ndvi.shape)

In [ ]:
# The same three pixels as Part A, and their tables (years x composites) of OBSERVED values.
# In Part B we never test values that were filled: a filled value is "normal" by construction.
# The pixels you chose in Part A (A1) were saved in my_pixels.json
import os, json
if os.path.exists("my_pixels.json"):                 # the file written in section A1 of Part A
    with open("my_pixels.json") as f:                # "with" opens the file and closes it at the end
        saved = json.load(f)                         # a dictionary: name -> [row, col]
    PIXELS = {}
    for name in saved:
        PIXELS[name] = tuple(saved[name])            # back to (row, col)
    print("Pixels chosen in Part A:", PIXELS)
else:
    # without the file we fall back to automatic pixels: they may NOT match the classes (see Part A, A1)
    auto = tt.example_pixels(ndvi)
    PIXELS = {"seasonal crop": auto["seasonal crop"], "stable vegetation": auto["stable vegetation"],
              "river sandbar": auto["intermediate"]}
    print("my_pixels.json not found (run section A1 of Part A first): using automatic pixels", PIXELS)
COLOURS = {"seasonal crop": "#2B580C", "stable vegetation": "#7B4F9E", "river sandbar": "#D9732B"}

# For each pixel, its observed series as a table: one row per year, one column per composite (1...46)
TABLES = {}
for name in PIXELS:
    row, col = PIXELS[name]
    TABLES[name] = tt.year_period_table(ndvi[:, row, col], dates)
    print(f"{name:18s}: table of {TABLES[name].shape[0]} years x {TABLES[name].shape[1]} composites")

# Part B · Is something changing?

### B1 · From the anomaly to the trend, step by step
We follow one pixel through the three steps, so that every later method has a clear place:

* **Step 1 (Part A): remove S.** anomaly = observed − climatology. Left: $I + T + \varepsilon$.
* **Step 2: remove ε.** The mean anomaly of each year: the noise goes up and down and largely cancels. Left: $I + T$, one value per year. A line through those values (Sen's slope, B3) is **T**.
* **Step 3: remove T.** Each yearly value minus the line. Left: **I**, how far each year was from its trend.

In [ ]:
name = "seasonal crop"                                    # try also "stable vegetation" or "river sandbar"
table = TABLES[name]                                      # observed values: years x composites

# Step 1 - remove S: subtract the climatology (Part A)
clim = tt.climatology(table, ref_years=REF_YEARS)
anomaly_table = tt.anomalies(table, clim)                 # absolute anomaly a, years x composites
print("Step 1 - anomaly of the first 5 composites of 2001:", anomaly_table.loc[2001].values[:5].round(3))

In [ ]:
# Step 2 - remove the noise: the mean of each ROW of the anomaly table (one value per year).
# min_share=0.6: a year needs at least 60 % of its 46 composites with data, otherwise it is left out (NaN)
yearly = tt.annual_aggregate(anomaly_table, how="mean", min_share=0.6)
print("Step 2 - mean anomaly of each year (I + T):")
print(yearly.round(3).to_string())

In [ ]:
# The line through the yearly values: Sen's slope (explained in B3)
line = tt.sens_slope(yearly.values, yearly.index.values)            # x = the years, y = the yearly values
# the value of the line in each year: intercept + slope x year
trend_line = line["intercept"] + line["slope"] * yearly.index.values
print(f"T: slope = {line['slope']:+.4f} NDVI per year = {10 * line['slope']:+.3f} NDVI per decade")

# Step 3 - remove T: what is left is I
I = yearly.values - trend_line
print("Step 3 - I (each year minus the line):", dict(zip(yearly.index, I.round(3))))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9))
# table_to_series puts the table back in time order (2001 composite 1, 2001 composite 2, ...)
tt.bars(axes[0], dates, tt.table_to_series(anomaly_table), width=7)
axes[0].set_title("Step 1 · remove S → anomaly (I + T + ε).  Q1: was this period normal?", loc="left")
tt.bars(axes[1], yearly.index, yearly.values)
axes[1].plot(yearly.index, trend_line, color="#7B4F9E", lw=2.5, label="Sen line (T)")
axes[1].legend(); axes[1].set_title("Step 2 · remove ε → one value per year (I + T).  Q2: is it changing?", loc="left")
tt.bars(axes[2], yearly.index, I)
axes[2].set_title("Step 3 · remove T → I.  Q3: which years were different?", loc="left")
for ax in axes: ax.set_ylabel("NDVI")
plt.tight_layout(); plt.show()

> 🧠 **Check your understanding.** Write your answers in a new markdown cell, in your own words. (1) Which years were furthest below their trend in this pixel? Do they coincide with El Niño (2002–03, 2009–10, 2015)? (2) Is the slope of step 2 large compared with the year-to-year differences of step 3? What does that tell you about how easy it is to detect this trend?

### B2 · Mann–Kendall, by hand
The test compares **every pair** of values (an earlier one and a later one) and counts +1 if the later is larger, −1 if smaller:

$$ S = \sum_{i<j} \operatorname{sign}(x_j - x_i), \qquad \operatorname{Var}(S) = \frac{n(n-1)(2n+5)}{18}, \qquad Z = \frac{S - \operatorname{sign}(S)}{\sqrt{\operatorname{Var}(S)}} $$

Without a trend, $S$ is close to 0. A large $|Z|$ is unlikely by chance (small p-value). It uses only the **order** of the values, not their size, so one extreme year does not move it.

The example of the lecture: five years of peak NDVI.

In [ ]:
x = np.array([0.60, 0.62, 0.61, 0.65, 0.64])      # five years of peak NDVI
n = len(x)                                        # number of years
S = 0
for i in range(n):                                # i = the earlier year
    for j in range(i + 1, n):                     # j = every LATER year (so each pair is counted once)
        # np.sign gives +1 if the later value is larger, -1 if it is smaller, 0 if equal
        sign = int(np.sign(x[j] - x[i]))
        S = S + sign                              # S adds up the +1 and -1 of all the pairs
        print(f"year {i + 1} -> year {j + 1}: {x[i]:.2f} -> {x[j]:.2f}   sign = {sign:+d}")
print("S =", S)

In [ ]:
var_S = n * (n - 1) * (2 * n + 5) / 18            # how much S varies by chance when there is no trend
# Z: how many standard deviations S is from 0 (the "- sign(S)" is a small continuity correction)
Z = (S - np.sign(S)) / np.sqrt(var_S)
print(f"Var(S) = {var_S:.2f}   Z = {Z:.2f}")
print("The toolbox:", tt.mann_kendall(x))

**Four of five years go up, and the test says "no evidence of a trend" (p ≈ 0.22).** With five values, chance produces a pattern like this quite often: **a short series cannot prove a trend**, however convincing it looks.

### B3 · Sen's slope, by hand

$$ \beta = \operatorname{median}\left\{ \frac{x_j - x_i}{t_j - t_i} : i < j \right\} $$

The **median** of the slopes of all pairs. Being a median, one extreme year barely moves it; a least-squares line can be tilted by one dry year at an end.

In [ ]:
slopes = []                                         # an empty list, filled pair by pair
for i in range(n):
    for j in range(i + 1, n):
        slopes.append((x[j] - x[i]) / (j - i))          # change per year between year i and year j
print("The 10 slopes, sorted:", np.round(sorted(slopes), 3))
print("Sen slope = their median =", round(np.median(slopes), 4), "NDVI per year")

In [ ]:
# Robustness: make the LAST year very dry, and compare with a least-squares line
x_dry = x.copy()                                    # a copy, so x does not change
x_dry[-1] = 0.45                                    # [-1] = the last element
# np.polyfit(x, y, 1) fits a straight line by least squares; [0] is its slope
least_squares = np.polyfit(np.arange(n), x_dry, 1)[0]
print(f"least-squares slope: {least_squares:+.4f}    Sen slope: {tt.sens_slope(x_dry)['slope']:+.4f}")

### B4 · An experiment: how often does the test lie?
At α = 0.05 the test should find a "trend" in **5 %** of series that have none. That promise holds only if the values are **independent**. NDVI values are not: each one resembles the previous one (**autocorrelation**), and wet and dry years shift all the composites of a year together.

The toolbox generates 300 synthetic series **without any trend** (15 years) and counts how many each method calls significant:

* **original**: Mann–Kendall on all the anomalies (15 × 23 = 345 values);
* **Hamed–Rao**: the same, with the variance of S corrected for autocorrelation (Hamed & Rao, 1998);
* **annual**: Mann–Kendall on one mean per year (15 values) — **step 2 of B1**.

⏳ About a minute.

In [ ]:
# Each call generates 300 synthetic series WITHOUT trend and returns, for each method,
# the share of series it calls "significant" (p < 0.05). The promise of the test is 0.05.
independent = tt.false_positive_experiment(ar1=0.0)                      # each value independent of the previous
autocorrelated = tt.false_positive_experiment(ar1=0.6)                   # each value resembles the previous one
with_years = tt.false_positive_experiment(ar1=0.6, interannual=0.02)     # plus wet and dry years

result = pd.DataFrame({"independent": independent, "autocorrelated": autocorrelated,
                       "autocorrelated + wet/dry years": with_years}).T
print("Share of series WITHOUT trend called significant (the promise is 0.05):")
print(result.round(3))

In [ ]:
result.plot(kind="bar", figsize=(9, 3.5), rot=0, title="False positives (dashed line = 5 %)")
plt.axhline(0.05, color="k", ls="--"); plt.ylabel("share of series"); plt.show()

> 🧠 **Check your understanding.** Write your answers in a new markdown cell, in your own words. (1) Which method keeps its promise in each situation? Why do wet and dry years act like autocorrelation for a test on all the composites? (2) The next cell repeats the experiment with series that DO have a real trend of 0.003 NDVI per year (`trend=0.003`). Which method detects it most often? What does the annual method lose in exchange for its honesty?

*The next cell does the computation the question needs: run it, and use its result in your answer.*

In [ ]:
print(tt.false_positive_experiment(ar1=0.6, interannual=0.02, trend=0.003, n_series=200))

### B5 · One pixel, three defensible routes
Three ways of testing the same pixel. They differ in how they remove the seasonal cycle and in how many values they test.

In [ ]:
name = "seasonal crop"                                        # try also the other two pixels
table = TABLES[name]

# Route 1 - anomalies of every composite + Hamed-Rao correction
# climatology -> anomalies (table) -> one long series of 690 values
anomaly_series = tt.table_to_series(tt.anomalies(table, tt.climatology(table, REF_YEARS)))
r1 = tt.mann_kendall_hamed_rao(anomaly_series)                  # test corrected for autocorrelation
# the slope comes per composite (one step = 8 days); x 46 converts it to per year
slope1 = tt.sens_slope(anomaly_series)["slope"] * P
print(f"Route 1, anomalies + Hamed-Rao: slope {10 * slope1:+.3f} NDVI/decade, p = {r1['p']:.3f}")

In [ ]:
# Route 2 - seasonal Mann-Kendall: composite 1 is compared only with composite 1 of other years, etc.
#           A peak is never compared with a trough, so the seasonal cycle does not need to be removed.
r2 = tt.seasonal_mann_kendall(table)
print(f"Route 2, seasonal Mann-Kendall: slope {10 * r2['slope']:+.3f} NDVI/decade, p = {r2['p']:.3f}")

In [ ]:
# Route 3 - one mean value per year (step 2 of B1)
annual = tt.annual_aggregate(table, "mean", min_share=0.6)        # one mean NDVI per year (15 values)
r3 = tt.mann_kendall_hamed_rao(annual.values)
slope3 = tt.sens_slope(annual.values, annual.index.values)["slope"]
print(f"Route 3, annual means         : slope {10 * slope3:+.3f} NDVI/decade, p = {r3['p']:.3f}")

The slopes are in **NDVI per decade** (per year × 10).

> 🧠 **Check your understanding.** Write your answers in a new markdown cell, in your own words. Do the three routes agree on the **sign**? On significance? If they disagree, which one would you report, and what would you write next to it?

### B6 · Two checks: the edges, and a slope that changes
**The edges.** A series that starts in a wet year and ends in a dry one can show a "trend" that only reflects those two years. We remove one or two years at each end and test again.

**Rolling Sen slope.** The slope in windows of 8 years that move one year at a time. If it crosses zero, the direction changed.

In [ ]:
# The same test after removing one or two years at the start and/or the end
for first, last in [(2001, 2015), (2002, 2015), (2001, 2014), (2002, 2014), (2003, 2013)]:
    part = annual.loc[first:last]                                 # .loc[a:b] keeps the years from a to b
    s = tt.sens_slope(part.values, part.index.values)["slope"]
    p_value = tt.mann_kendall_hamed_rao(part.values)["p"]
    print(f"{first}-{last}: slope {10 * s:+.3f} NDVI/decade, p = {p_value:.3f}")

In [ ]:
# One Sen slope per 8-year window (2001-2008, 2002-2009, ...), drawn at the central year of the window
rolling = tt.rolling_sen(annual, window=8)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(annual.index, annual.values, "o-", color=COLOURS[name])
axes[0].set_title(f"{name}: annual mean NDVI"); axes[0].set_ylabel("NDVI")
axes[1].plot(rolling.index, 10 * rolling.values, "o-", color="k")
axes[1].axhline(0, color="#C0392B")
axes[1].set_title("Sen slope in 8-year windows"); axes[1].set_xlabel("central year"); axes[1].set_ylabel("NDVI per decade")
plt.tight_layout(); plt.show()

> 🧠 **Check your understanding.** Write your answers in a new markdown cell, in your own words. (1) Does the sign survive when you remove years at the ends? (2) The next cell repeats the rolling slope with windows of 6, 8 and 10 years: what do you gain and lose with a shorter or a longer window?

*The next cell does the computation the question needs: run it, and use its result in your answer.*

In [ ]:
# The rolling Sen slope with three window lengths: 6, 8 and 10 years
fig, ax = plt.subplots(figsize=(10, 3.8))
for window, colour in [(6, "#D9732B"), (8, "k"), (10, "#2F6F8F")]:
    r_w = tt.rolling_sen(annual, window=window)               # one slope per window, at its central year
    ax.plot(r_w.index, 10 * r_w.values, "o-", color=colour, label=f"{window}-year windows ({len(r_w)} slopes)")
ax.axhline(0, color="#C0392B", lw=0.8)
ax.set_xlabel("central year"); ax.set_ylabel("NDVI per decade")
ax.set_title("Rolling Sen slope: shorter windows follow wet and dry years, longer ones are smoother but fewer", loc="left")
ax.legend(); plt.show()

### B7 · A map of trends, and the problem of thousands of tests
We test **every pixel** with route 3 (annual means, Hamed–Rao). With α = 0.05 and **no trend anywhere**, about 5 % of the pixels would still look "significant", and since neighbouring pixels resemble each other, those false positives form **patches that look real**.

The remedy: control the **False Discovery Rate** (Benjamini & Hochberg, 1995). Sort the p-values and accept only those with $p_{(k)} \le (k/m)\,\alpha$. Wilks (2016) recommends $\alpha_{FDR} = 0.10$ for maps. ⏳ One or two minutes.

In [ ]:
# One mean NDVI per pixel and year: a cube of (15 years, rows, columns)
annual_cube, years = tt.annual_mean_cube(ndvi, dates, min_share=0.6, steps_per_year=P)
# For EVERY pixel: Sen slope and Mann-Kendall p-value (Hamed-Rao); pixels with < 10 valid years are left NaN
slope, pval = tt.trend_map(annual_cube, years, method="hamed_rao", min_years=10)

tested = np.isfinite(pval).sum()                      # how many pixels were tested
naive = pval < 0.05                                   # True/False map without correction
fdr = tt.fdr_mask(pval, alpha=0.10)                   # True/False map after the FDR correction (Benjamini-Hochberg)
print("pixels tested:", tested)
print("'significant' with p < 0.05:", naive.sum(), f"({100 * naive.sum() / tested:.1f} %)")
print("significant after FDR:     ", fdr.sum(), f"({100 * fdr.sum() / tested:.1f} %)")
print("expected by chance alone if nothing changed: about", round(0.05 * tested))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# colour = Sen slope per decade (brown = decreasing, green = increasing); black dots = "significant" pixels
im = tt.show_trend_map(axes[0], slope, title="dots: p < 0.05, no correction", mask=naive)
tt.show_trend_map(axes[1], slope, pval, title="dots: significant after FDR (alpha = 0.10)")
plt.colorbar(im, ax=axes, shrink=0.8, label="Sen slope, NDVI per decade")
plt.show()

**How to report a map of trends:** the Sen slope in a colour scale centred on zero (NDVI per decade); dots only where the trend is significant **after** the correction; and in the legend: method, period, aggregation, autocorrelation correction and the α of the FDR.

> 🧠 **Check your understanding.** Write your answers in a new markdown cell, in your own words. (1) Where are the significant pixels after FDR? Do they match a land cover you recognise? (2) Give three reasons why a significant pixel might still not be a real change in vegetation.

### B8 · A slope or a step?
Mann–Kendall asks whether the series is **monotonic**. It cannot tell a slow change from a **step** in one year (a paddy turned into housing, a new canal, an uncorrected change of processing).

In [ ]:
# Two synthetic series without seasonal cycle (amplitude=0): a slow rise, and a jump of 0.1 in 2013
gradual = tt.synthetic_ndvi(trend_per_year=0.004, amplitude=0, interannual=0.015, noise=0.02, seed=3)
step = tt.synthetic_ndvi(trend_per_year=0.0, amplitude=0, interannual=0.015, noise=0.02, step_year=2013, step_size=0.1, seed=3)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for ax, label, d in [(axes[0], "gradual change", gradual), (axes[1], "one step in 2013", step)]:
    yearly_mean = d.groupby("year")["value"].mean()          # one mean value per year
    p_value = tt.mann_kendall(yearly_mean.values)["p"]
    ax.plot(yearly_mean.index, yearly_mean.values, "o-", color="k")
    ax.set_title(f"{label}: p = {p_value:.4f}"); ax.set_ylabel("NDVI")
plt.tight_layout(); plt.show()

Both are "significant increasing trends" for Mann–Kendall. Only **looking at the series**, or a method that searches for **breakpoints** (BFAST, Verbesselt et al., 2010), can tell them apart. Forkel et al. (2013): finding **too many** breakpoints is worse than too few; accept one when it coincides with something known.

### B9 · Two sensors, two periods: Sentinel-2 and MODIS
Sentinel-2 sees each paddy (20 m) but only since about 2017; MODIS mixes paddies and roads (250 m) but has a long record. To compare them over the **same years** we use two small files for the window of Session 1 (3.6 × 3.6 km): monthly **Sentinel-2 2018–2025** and **MODIS 2018–2025**. If the files are not in your folder, the functions build them once (⏳ Sentinel-2 takes several minutes per year).

Three maps with the same method:
* same years, different sensor → the effect of **resolution**;
* same sensor, different period (MODIS 2001–2015 vs 2018–2025) → a trend **belongs to its period**.

In [ ]:
# [west, south, east, north]: the 3.6 x 3.6 km window of Session 1 (0.0163 degrees ≈ 1.8 km each way)
S2_BBOX = [-74.985 - 0.0163, 3.915 - 0.0163, -74.985 + 0.0163, 3.915 + 0.0163]
# Each function reads its file if it exists; if not, it builds it once (Sentinel-2 is slow: minutes per year)
s2 = tt.get_s2_monthly_file("saldana_s2_monthly_2018_2025.h5", S2_BBOX, range(2018, 2026))
recent = tt.get_modis_file("saldana_modis_ndvi_2018_2025_window.h5", S2_BBOX, "2018-01-01/2025-12-31",
                           title="MODIS Terra + Aqua NDVI, Saldaña window, 2018-2025")
s2_dates = pd.DatetimeIndex(s2["time"])
recent_dates = pd.DatetimeIndex(recent["time"])
print("Sentinel-2:", s2["ndvi"].shape, "| MODIS 2018-2025:", recent["ndvi"].shape)

In [ ]:
# The MODIS 2001-2015 pixels that fall inside the same window
# rows and cols are True/False arrays: which MODIS rows/columns fall inside the Sentinel-2 window
rows, cols = tt.pixels_inside(cube, s2)
modis_window = ndvi[:, rows][:, :, cols]          # keep all the dates, only those rows, then only those columns
print("MODIS 2001-2015 pixels in the window:", modis_window.shape[1:], )

In [ ]:
# One map of trends per sensor and period (annual means, Hamed-Rao)
maps = {}                                         # label -> (slope map, p-value map)
# Sentinel-2: 12 monthly values per year; only 8 years, so we accept pixels with at least 6
a, yrs = tt.annual_mean_cube(s2["ndvi"], s2_dates, min_share=0.6, steps_per_year=12)
maps["Sentinel-2, 2018-2025"] = tt.trend_map(a, yrs, method="hamed_rao", min_years=6)
a, yrs = tt.annual_mean_cube(recent["ndvi"], recent_dates, min_share=0.6, steps_per_year=P)
maps["MODIS, 2018-2025"] = tt.trend_map(a, yrs, method="hamed_rao", min_years=6)
a, yrs = tt.annual_mean_cube(modis_window, dates, min_share=0.6, steps_per_year=P)
maps["MODIS, 2001-2015"] = tt.trend_map(a, yrs, method="hamed_rao", min_years=10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, label in zip(axes, maps):
    s, p = maps[label]
    im = tt.show_trend_map(ax, s, p, title=f"{label}\n{tt.fdr_mask(p).sum()} significant pixels after FDR", lim=0.2)
plt.colorbar(im, ax=axes, shrink=0.8, label="Sen slope, NDVI per decade")
plt.show()

**The same window, month by month.** NDVI from MODIS and Sentinel-2 is not identical (different bands, pixel size, compositing): expect a similar shape with an offset. That is why the two records cannot simply be joined.

In [ ]:
# The mean of the whole window for each date (axis=(1, 2) = rows and columns)
s2_mean = pd.Series(np.nanmean(s2["ndvi"], axis=(1, 2)), index=s2_dates)                         # one value per month
# MODIS has a value every 8 days: .resample("MS").mean() averages them by calendar month ("MS" = month start)
modis_mean = pd.Series(np.nanmean(recent["ndvi"], axis=(1, 2)), index=recent_dates).resample("MS").mean()

plt.figure(figsize=(14, 3.8))
plt.plot(s2_mean.index, s2_mean.values, "o-", ms=3, color="#2F6F8F", label="Sentinel-2 (20 m)")
plt.plot(modis_mean.index, modis_mean.values, "s-", ms=3, color="#D9732B", label="MODIS (250 m)")
plt.ylabel("window-mean NDVI"); plt.legend(); plt.show()
print(f"mean difference Sentinel-2 - MODIS: {(s2_mean - modis_mean).mean():+.3f} NDVI")

> 🧠 **Check your understanding.** Write your answers in a new markdown cell, in your own words. (1) Do Sentinel-2 and MODIS 2018–2025 agree on where NDVI rises or falls? What can Sentinel-2 show that MODIS cannot? (2) Does MODIS 2018–2025 keep the direction of MODIS 2001–2015? Which map answers "did the district become greener in the 2000s?", and which "which paddies changed recently?" (3) Why can't we join the MODIS series before 2018 with the Sentinel-2 series after it?

### B10 · Save the result with its recipe
A map of trends travels with its decisions (Session 1).

In [ ]:
import h5py
with h5py.File(TREND_FILE, "w") as f:            # "w" = create the file (replace it if it exists)
    # attributes: the recipe of the result, so that anyone can reproduce or question it
    f.attrs["title"] = "NDVI trends, Saldaña, MODIS Terra + Aqua 2001-2015"
    f.attrs["derived_from"] = MODIS_FILE
    f.attrs["aggregation"] = "annual mean NDVI (years with >= 60 % reliable composites)"
    f.attrs["test"] = "Mann-Kendall with Hamed & Rao (1998) correction"
    f.attrs["slope"] = "Theil-Sen, NDVI per year"
    f.attrs["multiple_testing"] = "Benjamini-Hochberg FDR, alpha = 0.10"
    # datasets: the maps themselves (float32 = half the size of float64, enough precision for NDVI)
    f.create_dataset("sen_slope_per_year", data=slope.astype("float32"))
    f.create_dataset("p_value", data=pval.astype("float32"))
    f.create_dataset("significant_fdr", data=fdr.astype("uint8"))
    f.create_dataset("x", data=cube["x"]); f.create_dataset("y", data=cube["y"])
tt.describe_hdf5(TREND_FILE)

### B11 · A bridge to Session 3: derivatives and integrals
* The **integral** of a season turns its shape into one number per year (intra-annual → inter-annual).
* The **first derivative** (rate of change) says when the vegetation grows fastest (green-up) and falls fastest (harvest). Savitzky–Golay gives it with `deriv=1` (Session 1).

In [ ]:
from scipy.signal import savgol_filter
row, col = PIXELS["seasonal crop"]
# the series of the paddy, short gaps (<= 2 values) interpolated, only 2012-2014, without the remaining NaN
s = pd.Series(ndvi[:, row, col], index=dates).interpolate(limit=2).loc["2012":"2014"].dropna()
smooth = savgol_filter(s.values, 9, 2)                           # Savitzky-Golay: window of 9 values, parabola
# deriv=1 gives the first derivative (change per step); one step = 8 days, so / 8 gives NDVI per day
rate = savgol_filter(s.values, 9, 2, deriv=1) / 8

fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
axes[0].plot(s.index, s.values, ".", color="#888888"); axes[0].plot(s.index, smooth, color="#2B580C")
axes[0].set_ylabel("NDVI")
axes[1].plot(s.index, rate, color="k"); axes[1].axhline(0, color="#C0392B")
axes[1].set_ylabel("NDVI per day"); axes[1].set_title("first derivative: + green-up, - senescence or harvest", loc="left")
plt.tight_layout(); plt.show()

## 🌎 Apply it to your own study area — Part B
Repeat Part B with your own study area: at the top of this notebook write in `MODIS_FILE` the same file name you used in Part A (and change `TREND_FILE`, so the result of Saldaña is not overwritten). Part B reads the three pixels you chose in A1 from `my_pixels.json`. No new code is needed: the cells are the same. Then write, in your own words:

1. **Report one pixel.** For one of your pixels, report the trend with two routes (B5): slope per decade with its interval, p-value and method, and one sentence on whether the edges (B6) change the conclusion. The next cell prints the numbers you need.
2. **The legend of a map.** Write the complete legend of your map of trends (B7) as you would in a thesis: method, period, aggregation, autocorrelation correction and the α of the FDR.
3. **Is it real?** Choose one patch of significant pixels in your map and go through the traps of the study guide (section 9): could it come from the edges, the sensor, a step or a mixed pixel?

*The next cell does the computation the question needs: run it, and use its result in your answer.*

In [ ]:
table = TABLES["seasonal crop"]                       # change it to your own pixel
annual = tt.annual_aggregate(table, "mean", min_share=0.6)
s = tt.sens_slope(annual.values, annual.index.values)   # contains slope, intercept and its interval (low, high)
r = tt.mann_kendall_hamed_rao(annual.values)
print(f"Annual mean, MK + Hamed-Rao: {10 * s['slope']:+.3f} NDVI/decade [{10 * s['low']:+.3f}, {10 * s['high']:+.3f}], p = {r['p']:.3f}")
seas = tt.seasonal_mann_kendall(table)
print(f"Seasonal MK: {10 * seas['slope']:+.3f} NDVI/decade, p = {seas['p']:.3f}")

## 🧠 Self-check · Part B
Answer without running code, in your own words. If you cannot answer one, go back to the section in brackets.

1. After step 2 there is one value per year. What was removed, what is left, and why does averaging the 46 anomalies of a year remove the noise but not $I$ or $T$? (B1)
2. Five years of peak NDVI: 0.60, 0.62, 0.61, 0.65, 0.64. Compute Mann–Kendall's $S$ by hand. Why is the result not significant, although four of five years go up? (B2)
3. What is the difference between the question Mann–Kendall answers and the one Sen's slope answers? Why are both reported? (B2, B3)
4. Explain how the Theil–Sen line is estimated: how many pairs are there with 15 years, what is computed for each pair, and how are the slope and the intercept obtained? (B3)
5. Why does one very dry last year tilt a least-squares line but hardly move the Sen line? (B3)
6. A test at α = 0.05 should call "significant" 5 % of the series without trend. Why does it call many more when the values are autocorrelated? Give two sources of autocorrelation in our NDVI series. (B4)
7. Why is the test on one value per year the most honest one, and what does it lose in exchange? (B4, B5)
8. The three routes of B5 give slopes of the same sign but only one is significant. What do you report, and what must you not do? (B5)
9. Removing the first two years changes the sign of the slope. What does that tell you about the trend? (B6)
10. In a map of 5 000 pixels without any real trend, how many would come out "significant" at p < 0.05? What does the FDR correction do about it? (B7)
11. A series has a step in 2013 (a paddy turned into houses). What does Mann–Kendall say, and why is that misleading? (B8)
12. Sentinel-2 2018–2025 and MODIS 2001–2015 give different slopes for the same window. Is that a contradiction? Which sensor answers which question? (B9)

## Synthesis
1. A series is a sum of parts at different time scales; each question removes parts **in order**: S (anomaly), then ε (one value per year), then T (what is left is I).
2. **Moving averages** separate scales; a one-year average removes the season. **Classical decomposition** is rigid; **STL** is flexible, but its trend describes, it does not test.
3. An **anomaly** is observed − expected for that time of year; **standardise** it with a robust spread to compare pixels; in crops it mixes amount and **timing**.
4. **Mann–Kendall**: is it monotonic? **Sen**: how much? Report both, with the method.
5. Autocorrelation, short records, edges, sensors and thousands of tests produce **false trends**: aggregate, correct, check the edges, use FDR, look at the series.

**Next session:** phenology — when each season starts, peaks and ends.

## End of Week 10
Part A asked **was it normal?** (step 1: the anomaly). Part B asked **is it changing?** (step 2: one value per year and the Sen line, with Mann–Kendall) and **which years were different?** (step 3). Both notebooks are templates: repeat them with your own study area, and answer the 🧠 questions in your own words.